## Part 5: Introduction to Machine Learning and Logistic Regression

So far, we have focused on analyzing the outputs of a facial recognition system by measuring accuracy and error across different groups. But where do these predictions come from?

### What is machine learning?

Many AI systems use something called **machine learning**. Instead of following a fixed set of rules, a machine learning model learns patterns from data.

For example, if we show a model many examples of inputs and their correct answers, it can start to learn how to make predictions on new data. This process is often called **supervised learning**, where the model learns from labeled examples (data where we already know the correct answer).

In this tutorial, the model takes in numerical features derived from facial images and tries to predict a label based on those features.

### What is logistic regression?

One simple method we can use is **logistic regression**. Even though it has "regression" in its name, it is commonly used for classification tasks.

Logistic regression works by looking at the input features and combining them using learned weights. It then converts this result into a number between 0 and 1, which we can think of as a probability.

For example, the model might output something like 0.8, meaning it is 80% confident in a certain prediction. Based on this value, the model decides which class the input belongs to.

### What does our logistic regression model code do?

In this tutorial, we use logistic regression as our machine learning model to make predictions based on facial image features.

The model follows a simple pipeline:

1. Load data from a CSV file  
2. Select input features  
3. Select the target labels  
4. Convert labels into numbers  
5. Train a logistic regression model  

First, let’s import the libraries we need.

- pandas: used to read and organize data  
- LogisticRegression: the machine learning model  
- LabelEncoder: converts text labels into numbers  

In [13]:
import os
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score

Next, we specify the dataset and which columns we want to use.

The model needs:

- feature columns: numerical inputs used by the model  
- target column: the label the model is trying to predict  

In [20]:
dataset_path = "datasets/"

datasets = {
    "A": "train_A_emb.csv",
    "B": "train_B_emb.csv",
    "C": "train_C_emb.csv"
}

Now, we load the dataset and separate it into inputs and labels.

In [15]:
def load_data(path):
    df = pd.read_csv(path)

    # Features (embeddings)
    feature_cols = df.columns[:128]
    X = df[feature_cols].values

    # Target (what we predict)
    y = df["race"].values

    # Extra attribute (for bias analysis later)
    gender = df["gender"].values

    return X, y, gender, df

Machine learning models require numerical inputs, so we convert labels like "Male" and "Female" into numbers.

In [16]:
def encode_labels(labels):
    # Convert labels like male/female into numbers (0,1)
    le = LabelEncoder()
    encoded = le.fit_transform(labels)
    return le, encoded

Next, we create and train the logistic regression model.
During training, the model learns patterns from the data by adjusting its weights.

In [17]:
def train(input_data, target_encoded):
    # Create the logistic regression model
    model = LogisticRegression(max_iter=1000, random_state=42)
    
    # Train the model
    model.fit(input_data, target_encoded)
    
    return model

Now, we create the model for evaluation of our trained model. Phrase this in your cool tutorial way lol

In [18]:
def evaluate_model(model, X_test, y_test, df_test):
    y_pred = model.predict(X_test)

    # Overall accuracy
    overall_acc = accuracy_score(y_test, y_pred)

    # Add predictions to dataframe
    df_test = df_test.copy()
    df_test["pred"] = y_pred

    # Accuracy by race
    race_acc = (df_test["pred"] == df_test["race"]) \
        .groupby(df_test["race"]) \
        .mean()

    # Accuracy by gender
    gender_acc = (df_test["pred"] == df_test["race"]) \
        .groupby(df_test["gender"]) \
        .mean()

    return overall_acc, race_acc, gender_acc, y_pred

Finally, we put everything together.

This loads the data, converts the labels, and trains the model.

In [21]:
# Load test set once
X_test, y_test, gender_test, test_df = load_data(dataset_path + "test_emb.csv")

models = {}
label_encoders = {}
preds = {}

for name, file in datasets.items():
    # Load training data
    X_train, y_train, _, _ = load_data(dataset_path + file)

    # Encode labels
    le, y_train_enc = encode_labels(y_train)

    # Train model
    model = train(X_train, y_train_enc)

    # Store
    models[name] = model
    label_encoders[name] = le

    # Predict (decode back to labels)
    y_pred_enc = model.predict(X_test)
    y_pred = le.inverse_transform(y_pred_enc)

    preds[name] = y_pred

In [23]:
for name in datasets.keys():
    print(f"\n--- Model {name} ---")

    temp_df = test_df.copy()
    temp_df["pred"] = preds[name]

    # Overall
    overall_acc = accuracy_score(y_test, preds[name])
    print("Overall Accuracy:", overall_acc)

    # By race
    race_acc = (temp_df["pred"] == temp_df["race"]) \
        .groupby(temp_df["race"]) \
        .mean()
    print("\nAccuracy by Race:")
    print(race_acc)

    # By gender
    gender_acc = (temp_df["pred"] == temp_df["race"]) \
        .groupby(temp_df["gender"]) \
        .mean()
    print("\nAccuracy by Gender:")
    print(gender_acc)


--- Model A ---
Overall Accuracy: 0.4107142857142857

Accuracy by Race:
race
Black              0.625
East Asian         0.575
Indian             0.350
Latino_Hispanic    0.225
Middle Eastern     0.300
Southeast Asian    0.400
White              0.400
dtype: float64

Accuracy by Gender:
gender
Female    0.433824
Male      0.388889
dtype: float64

--- Model B ---
Overall Accuracy: 0.3964285714285714

Accuracy by Race:
race
Black              0.500
East Asian         0.550
Indian             0.350
Latino_Hispanic    0.125
Middle Eastern     0.175
Southeast Asian    0.275
White              0.800
dtype: float64

Accuracy by Gender:
gender
Female    0.404412
Male      0.388889
dtype: float64

--- Model C ---
Overall Accuracy: 0.31785714285714284

Accuracy by Race:
race
Black              0.350
East Asian         0.400
Indian             0.300
Latino_Hispanic    0.050
Middle Eastern     0.075
Southeast Asian    0.225
White              0.825
dtype: float64

Accuracy by Gender:
gender
Femal